# SED Builder Demo 

The `ssdc-sedbuilder` package provides programmatic access to multi-wavelength spectral energy distribution (SED) data from catalogs, surveys, and archival observations across the electromagnetic spectrum. It is based on [SED Builder](https://tools.ssdc.asi.it/SED/), a software developed at ASI-SSDC to produce and display the SED data over the web. The [source code](https://gitrepo.ssdc.asi.it/sedbuilder/sedbuilder) and the [documentation](https://peppedilillo.github.io/sedbuilder/development/) are available online.

## Fetching data with `get_data`

The most important function of the package is called `get_data`. It queries the SSDC SED Builder API to retrieve Spectral Energy Distribution data from the main database. Data can be searched by coordinates:

In [1]:
from sedbuilder import get_data

response = get_data(ra=166.1138 , dec=38.2083)

> Note: angles are assumed in degrees. 

You can also search by the astronomical source name:

In [2]:
response = get_data(name="MRK 421")

> Note: `get_data` is _keyword only_: calling `get_data("MRK 421")` or `get_data(166.1138, 38.2083)` will result in an error, you must state the keyword name explicitly.


When `get_data` is called with the `name` argument, it resolves the source name to coordinates via SSDC name resolver, another API service offered by SSDC. You can query the SSDC name resolver directly with `resolve_name`. Calling `get_data` by coordinates is a bit faster, keep it in mind if you are going to query many sources in a loop.

In [3]:
from sedbuilder import resolve_name

(ra, dec), db = resolve_name("MRK 421")
print(f"Resolved 'MRK 421' to coordinates ra = {ra:.4f} deg and dec = {dec:.4f} deg, through the {db} database.")

Resolved 'MRK 421' to coordinates ra = 166.1138 deg and dec = 38.2088 deg, through the SSDC database.


## The `Response` object

The `get_data` function outputs a `Response` object.  What's inside a SED Builder `Response`? 

Data, of course! You can access it in multiple tabular formats, for example pandas:

In [4]:
df = response.to_pandas()
df

,name,frequency,nufnu,frequency_error,nufnu_error,angular_distance,start_time,stop_time,info,reference_name,reference_band,error_radius
0,None,8.460000e+09,5.343336e-14,0.000000e+00,3.384000e-17,0.000730,47941.500000,47941.500000,,CLASSSCAT,Radio,1.00
1,J110428+381228,8.400000e+09,5.305440e-14,0.000000e+00,0.000000e+00,0.000725,NaN,NaN,,CRATES,Radio,1.00
2,B2.3 1101+38,4.080000e+08,4.692000e-15,0.000000e+00,0.000000e+00,0.214886,NaN,NaN,,DIXON,Radio,1.00
3,S4 1101+38,2.700000e+09,2.079000e-14,0.000000e+00,0.000000e+00,0.161011,NaN,NaN,,DIXON,Radio,1.00
4,S4 1101+38,1.070000e+10,8.453000e-14,0.000000e+00,0.000000e+00,0.161011,NaN,NaN,,DIXON,Radio,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...
2296,None,8.850000e+18,2.147220e-11,2.079000e+17,4.999570e-12,0.000107,57839.910498,57840.639699,,NuBlazar,None,0.15
2297,None,9.271000e+18,2.877670e-11,2.128000e+17,6.012190e-12,0.000107,57839.910498,57840.639699,,NuBlazar,None,0.15
2298,None,9.764000e+18,1.663020e-11,2.805000e+17,5.108790e-12,0.000107,57839.910498,57840.639699,,NuBlazar,None,0.15
2299,None,1.034000e+19,1.980570e-11,2.950000e+17,5.911410e-12,0.000107,57839.910498,57840.639699,,NuBlazar,None,0.15


Or astropy tables:

In [5]:
at = response.to_astropy()
at[:5] # we only show the first five rows

name,frequency,nufnu,frequency_error,nufnu_error,angular_distance,start_time,stop_time,info,reference_name,reference_band,error_radius
,Hz,erg / (s cm2),Hz,erg / (s cm2),arcsec,d,d,,,,arcsec
str23,float64,float64,float64,float64,float64,float64,float64,str39,str37,str10,float64
None,8460000000.0,5.343335955180994e-14,0.0,3.3840001528216023e-17,0.0007296104773448408,47941.5,47941.5,,CLASSSCAT,Radio,1.0
J110428+381228,8400000000.0,5.305440000000001e-14,0.0,0.0,0.000725056715106309,nan,nan,,CRATES,Radio,1.0
B2.3 1101+38,408000000.0,4.6920000000000005e-15,0.0,0.0,0.21488626928846333,nan,nan,,DIXON,Radio,1.0
S4 1101+38,2700000000.0,2.0790000000000002e-14,0.0,0.0,0.1610107950611066,nan,nan,,DIXON,Radio,1.0
S4 1101+38,10700000000.0,8.453000000000001e-14,0.0,0.0,0.1610107950611066,nan,nan,,DIXON,Radio,1.0


The response method `to_jetset` make it easy working with [JetSeT](https://github.com/andreatramacere/jetset), a popular astrophysics framework for modelling radiative and accelerative processes.

In [6]:
jt = response.to_jetset(z=0.0308) # note that you must provide a redshift value when calling `to_jetset`
jt[:5]

x,dx,y,dy,T_start,T_stop,UL,dataset
Hz,Hz,erg / (s cm2),erg / (s cm2),,,,
float64,float64,float64,float64,float64,float64,bool,str37
8460000000.0,0.0,5.343335955180994e-14,3.3840001528216023e-17,47941.5,47941.5,False,CLASSSCAT
8400000000.0,0.0,5.305440000000001e-14,0.0,0.0,0.0,False,CRATES
408000000.0,0.0,4.6920000000000005e-15,0.0,0.0,0.0,False,DIXON
2700000000.0,0.0,2.0790000000000002e-14,0.0,0.0,0.0,False,DIXON
10700000000.0,0.0,8.453000000000001e-14,0.0,0.0,0.0,False,DIXON


You can also output data from a response into a Python dict with `response.to_dict()` or into JSON with `response.to_json()`.

A response also contains metadata, for example units:

In [7]:
from pprint import pprint

pprint(response.properties.units)

{'AngularDistance': 'arcsec',
 'ErrorRadius': 'arcsec',
 'Frequency': 'Hz',
 'FrequencyError': 'Hz',
 'Nh': 'cm**-2',
 'Nufnu': 'erg cm**(-2) s**(-1)',
 'NufnuError': 'erg cm**(-2) s**(-1)',
 'StartTime': 'mjd',
 'StopTime': 'mjd'}


You may have noticed a few columns having `reference` in their name. A reference is either a catalog or paper from which SED data come from. They are accessible through the `references` method:

In [8]:
for ref in response.references():
    if ref["name"] == "CLASSSCAT":
        pprint(ref)
        break

{'band': 'Radio',
 'bibliography': [{'authors': 'Myers, S. T.; Jackson, N. J.; Browne I. W. A. '
                              'et al.',
                   'id': None,
                   'title': 'The Cosmic Lens All Sky Survey - I. Source '
                            'selection and observations',
                   'url': 'https://ui.adsabs.harvard.edu/abs/2003MNRAS.341....1M'}],
 'error_radius': 1.0,
 'id': 4143,
 'kind': 'Catalog',
 'name': 'CLASSSCAT'}
